utils

In [ ]:
import uuid
from datetime import date
from functools import reduce
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F

def _as_list(x):
    return [x] if isinstance(x, str) else list(x)

def _for_write(df, spark, table_name):
    """Control the number of output files before writing to Hive.
    Partitioned table     -> all rows of one partition go to one task (1 file per partition per run).
    Non-partitioned table -> a single file (fine for small tables)."""
    db_name, _, tbl = table_name.rpartition('.')
    pcols = [c.name for c in (spark.catalog.listColumns(tbl, db_name) if db_name
                              else spark.catalog.listColumns(tbl)) if c.isPartition]
    return df.repartition(*pcols) if pcols else df.coalesce(1)

def _add_scd_columns(df: DataFrame, today, first_start_col: str = None) -> DataFrame:
    """
    Add start_date / end_date / is_current.
    start_date = today, unless first_start_col is given: then it is the date in that column
    (e.g. the user's registration date), falling back to today when that date is missing.
    """
    start = F.lit(today).cast("date")
    if first_start_col:
        start = F.coalesce(F.col(first_start_col).cast("date"), start)
    return (
        df.withColumn("start_date", start)
          .withColumn("end_date", F.lit(None).cast("date"))
          .withColumn("is_current", F.lit(True))
    )

def _assign_surrogate_key(df: DataFrame, surrogate_key: str, keys: list, offset: int) -> DataFrame:
    """Hive has no auto-increment, so generate surrogate keys = offset + row_number"""
    w = Window.orderBy(*keys)
    return df.withColumn(surrogate_key, (F.row_number().over(w) + F.lit(offset)).cast("long"))

def _partition_condition(pcols: list, partition_tuples: list):
    """Column expression that keeps only rows living in the given partitions (a list of value tuples)."""
    cond = None
    for t in partition_tuples:
        piece = None
        for name, val in zip(pcols, t):
            p = F.col(name).isNull() if val is None else (F.col(name) == F.lit(val))
            piece = p if piece is None else (piece & p)
        cond = piece if cond is None else (cond | piece)
    return cond

def _prepare_hive_writes(spark: SparkSession, overwrite_mode: str):
    """
    Session settings every Hive write in this module relies on.
    overwrite_mode: 'static'  -> INSERT OVERWRITE replaces the whole table (full rebuilds, SCD2)
                    'dynamic' -> INSERT OVERWRITE replaces only the partitions present in the data (upserts)
    Set explicitly on every call so one function can never inherit the other's mode.
    """
    spark.sql("SET hive.exec.dynamic.partition=true")
    spark.sql("SET hive.exec.dynamic.partition.mode=nonstrict")
    spark.sql(f"SET spark.sql.sources.partitionOverwriteMode={overwrite_mode}")

scd_type 2_load

In [ ]:
def scd2_load(
    df: DataFrame,
    table_name: str,
    business_key,
    tracked_cols: list,
    spark: SparkSession,
    surrogate_key: str = None,
    first_start_col: str = None,
):
    """
    Generic SCD Type 2 loader for Hive tables using Spark DataFrames.

    Parameters
    ----------
    df            : transformed Spark DataFrame ready to load
    table_name    : target Hive table (db.table) 
    business_key  : natural key column, or list of columns (composite key)
    tracked_cols  : list of columns to monitor for changes
    spark         : SparkSession with Hive support enabled
    surrogate_key : optional surrogate key column in the target table. If given,
                    new rows get max(existing) + row_number(). Leave as None if the
                    table has no surrogate key or df already supplies one.
    first_start_col: optional column holding the date a record FIRST existed in the real
                    world (e.g. the registration date). A brand-new record's first version
                    starts on that date instead of the load date, so older facts can still
                    find the version that was valid at their date. Later versions (after a
                    change) always start on the load date. The column is not loaded.

    Notes
    -----
    Plain Hive tables don't support UPDATE, so expiring old rows is done by
    rebuilding the full table content (history + expired + new versions) and
    overwriting the table. The rebuilt data is staged in a temporary table first,
    because Spark can't overwrite a table it is reading from in the same job.
    """
    today = date.today()
    keys = _as_list(business_key)

    # ── Input validation ─────────────────────────────────────────
    if not tracked_cols:
        raise ValueError(f"{table_name}: tracked_cols must not be empty")

    missing = [c for c in tracked_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name}: tracked_cols not found in DataFrame: {missing}")

    if first_start_col and first_start_col not in df.columns:
        raise ValueError(f"{table_name}: first_start_col '{first_start_col}' not found in DataFrame")

    for k in keys:
        if k not in df.columns:
            raise ValueError(f"{table_name}: business key '{k}' not found in DataFrame")
        n_null = df.filter(F.col(k).isNull()).count()
        if n_null > 0:
            raise ValueError(f"{table_name}: {n_null} null values in business key '{k}'")

    # A full rebuild must replace the whole table, never just some partitions
    _prepare_hive_writes(spark, "static")

    # ── Dedup: one row per business key ──────────────────────────
    df = df.dropDuplicates(keys)

    target = spark.table(table_name)
    target_cols = target.columns

    # ── FIRST LOAD ──
    if len(target.limit(1).collect()) == 0:
        print(f"First load detected — inserting all records into {table_name}.")

        rows = _add_scd_columns(df, today, first_start_col)
        if surrogate_key:
            rows = _assign_surrogate_key(rows, surrogate_key, keys, offset=0)

        rows = rows.select(*target_cols).cache()

        # SAFETY: column order must match Hive exactly for insertInto
        assert rows.columns == target_cols, (
            f"Column mismatch for {table_name}\n"
            f"  DF    : {rows.columns}\n"
            f"  Target: {target_cols}"
        )

        try:
            n = rows.count()
            _for_write(rows, spark, table_name).write.mode("append").insertInto(table_name)
        finally:
            rows.unpersist()

        print(f"Inserted {n} records.")
        print(f"{table_name} SCD Type II load completed.")
        return

    # ── SUBSEQUENT LOAD ──
    existing_current = target.filter(F.col("is_current") == True) 

    old = existing_current.select(
        *keys,
        *[F.col(c).alias(f"{c}__old") for c in tracked_cols],
    ).withColumn("_in_dw", F.lit(True))

    joined = df.join(old, on=keys, how="left")

    # Null-safe comparison on string representations (NULL == NULL is "unchanged")
    changed_cond = reduce(
        lambda a, b: a | b,
        [
            ~F.col(c).cast("string").eqNullSafe(F.col(f"{c}__old").cast("string"))
            for c in tracked_cols
        ],
    )

    new_df = joined.filter(F.col("_in_dw").isNull()).select(*df.columns).cache()
    changed_df = (
        joined.filter(F.col("_in_dw").isNotNull() & changed_cond)
              .select(*df.columns)
              .cache()
    )

    stg_table = f"{table_name}__scd2_stg_{uuid.uuid4().hex[:8]}"

    try:
        n_new, n_changed, n_total = new_df.count(), changed_df.count(), df.count()

        print(f"New records:     {n_new}")
        print(f"Changed records: {n_changed}")
        print(f"Unchanged:       {n_total - n_new - n_changed}")

        if n_new == 0 and n_changed == 0:
            print(f"{table_name} — nothing to load.")
            print(f"{table_name} SCD Type II load completed.")
            return

        # Expire changed records (current version only)
        expire_keys = (
            changed_df.select(*keys).distinct().withColumn("_expire", F.lit(True))
        )
        to_expire = F.col("_expire").isNotNull() & (F.col("is_current") == True)  # noqa: E712

        existing_all = (
            target.join(expire_keys, on=keys, how="left")
                  .withColumn(
                      "end_date",
                      F.when(to_expire, F.lit(today).cast("date")).otherwise(F.col("end_date")),
                  )
                  .withColumn(
                      "is_current",
                      F.when(to_expire, F.lit(False)).otherwise(F.col("is_current")),
                  )
                  .drop("_expire")
                  .select(*target_cols)
        )
        print(f"Expired {n_changed} old records.")

        # New versions: a brand-new record starts on first_start_col (if given),
        # a changed record starts today (the day the change was noticed)
        to_insert = (_add_scd_columns(new_df, today, first_start_col)
                     .unionByName(_add_scd_columns(changed_df, today)))

        if surrogate_key:
            max_sk = target.agg(F.max(surrogate_key)).collect()[0][0] or 0
            to_insert = _assign_surrogate_key(to_insert, surrogate_key, keys, offset=max_sk)

        to_insert = to_insert.select(*target_cols)

        result = existing_all.unionByName(to_insert)

        # SAFETY: enforce column order before staging/writing
        assert result.columns == target_cols, (
            f"Column mismatch for {table_name}\n"
            f"  DF    : {result.columns}\n"
            f"  Target: {target_cols}"
        )

        # Stage → overwrite target → drop stage
        result.write.mode("overwrite").saveAsTable(stg_table)

        (
            _for_write(spark.table(stg_table).select(*target_cols), spark, table_name)
                 .write.mode("overwrite")
                 .insertInto(table_name)
        )

        print(f"Inserted {n_new + n_changed} new/updated records.")
        print(f"{table_name} SCD Type II load completed.")
    finally:
        # Always clean up, even if the load failed half-way
        spark.sql(f"DROP TABLE IF EXISTS {stg_table}")
        new_df.unpersist()
        changed_df.unpersist()

simple_load

In [ ]:
def simple_load(df: DataFrame, table_name: str, business_key, spark: SparkSession, partition_cols=None):
    """
    Smart simple load — inserts only NEW records into a Hive table.
    No SCD, no history, and nothing is ever overwritten: rows already in the table are never touched.
    That is why it is safe for a table that several jobs write to at the same time (e.g. fact_events).

    Parameters
    ----------
    df            : transformed Spark DataFrame ready to load
    table_name    : target Hive table (db.table)
    business_key  : column or list of columns to detect new records
    spark         : SparkSession with Hive support enabled
    partition_cols: optional, for a partitioned table. Only the partitions present in the incoming
                    data are searched for existing keys, instead of reading every key in the table.
                    Assumes a given key always lives in the same partition (true for event ids,
                    because the partition comes from the event's own timestamp).
    """
    keys = _as_list(business_key)

    for k in keys:
        if k not in df.columns:
            raise ValueError(f"{table_name}: business key '{k}' not found in DataFrame")

    _prepare_hive_writes(spark, "static")

    df = df.dropDuplicates(keys).cache()

    try:
        target = spark.table(table_name)
        target_cols = target.columns

        n_incoming = df.count()
        print(f"Incoming records : {n_incoming}")
        if n_incoming == 0:
            print(f"{table_name} — empty batch, nothing to insert.")
            return

        if partition_cols:
            pcols = _as_list(partition_cols)
            touched = [tuple(r) for r in df.select(*pcols).distinct().collect()]
            existing_keys = target.filter(_partition_condition(pcols, touched)).select(*keys)
        else:
            existing_keys = target.select(*keys)
            print(f"Existing records : {existing_keys.count()}")

        # Records not in the DW yet
        new_records = df.join(existing_keys, on=keys, how="left_anti").cache()

        try:
            n_new = new_records.count()
            print(f"New records      : {n_new}")

            if n_new == 0:
                print(f"{table_name} — no new records to insert.")
            else:
                _for_write(new_records.select(*target_cols), spark, table_name).write.mode("append").insertInto(table_name)
                print(f"Inserted {n_new} new records into {table_name}.")
        finally:
            new_records.unpersist()
    finally:
        df.unpersist()

    print(f"{table_name} load completed.")

upsert_fact

In [ ]:
def upsert_fact(df, table_name, business_key, partition_cols, spark):
    """
    Idempotent upsert for a partitioned Hive fact table.

    For each partition touched by the incoming batch:
        new partition content = (target rows in that partition NOT in incoming)
                                ∪ (incoming batch)
    Other partitions are left untouched.

    The merged result is staged in a temporary table before the overwrite,
    because Spark can't overwrite a table it is still reading from.
    """
    _prepare_hive_writes(spark, "dynamic")

    keys  = _as_list(business_key)
    pcols = _as_list(partition_cols)

    df = df.dropDuplicates(keys).cache()
    stg_table = f"{table_name}__upsert_stg_{uuid.uuid4().hex[:8]}"

    try:
        n_incoming = df.count()
        if n_incoming == 0:
            print(f"{table_name}: empty batch — nothing to upsert.")
            return

        target = spark.table(table_name)
        target_cols = target.columns
        incoming = df.select(*target_cols)

        # If table is empty, it's a first load — just append
        if len(target.limit(1).collect()) == 0:
            _for_write(incoming, spark, table_name).write.mode("append").insertInto(table_name)
            print(f"First load: inserted {n_incoming} rows into {table_name}.")
            return

        # Which partitions does the incoming batch touch?
        touched_tuples = [tuple(r) for r in df.select(*pcols).distinct().collect()]
        print(f"{table_name}: touched partitions = {touched_tuples}")

        # Build a filter for the touched partitions
        cond = None
        for t in touched_tuples:
            piece = None
            for name, val in zip(pcols, t):
                p = F.col(name).isNull() if val is None else (F.col(name) == F.lit(val))
                piece = p if piece is None else (piece & p)
            cond = piece if cond is None else (cond | piece)

        target_affected = target.filter(cond)

        # Keys present in incoming batch
        incoming_keys = df.select(*keys).distinct()

        # Target rows in affected partitions NOT present in incoming → keep as-is
        untouched = target_affected.join(incoming_keys, on=keys, how="left_anti")

        # Merge: untouched + incoming (already ordered like the target)
        merged = untouched.unionByName(incoming)

        assert merged.columns == target_cols, (
            f"Column mismatch for {table_name}\n"
            f"  DF    : {merged.columns}\n"
            f"  Target: {target_cols}"
        )

        # Stage → dynamic partition overwrite (only touched partitions are replaced)
        merged.write.mode("overwrite").saveAsTable(stg_table)

        (
            _for_write(spark.table(stg_table).select(*target_cols), spark, table_name)
                 .write.mode("overwrite")
                 .insertInto(table_name)
        )
        print(f"{table_name}: upsert complete "
              f"({n_incoming} incoming rows, affected partitions replaced, others untouched).")
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {stg_table}")
        df.unpersist()

attach_scd2_key (as-of join to an SCD2 dimension)

In [ ]:
def attach_scd2_key(df, ts_col, row_key, dim_table, business_key, surrogate_key, spark):
    """
    As-of join to an SCD2 dimension: give every row the surrogate key of the dimension VERSION that was
    valid on the row's date (start_date <= date < end_date; end_date empty = still current).
    Fallback if no version covers the date: the current version, then 0 (Unknown member).

    row_key is the fact's unique id; it makes sure exactly one version is picked per row.
    The dimension is read from Hive on every call, so new members and SCD2 changes are always seen
    (also inside a streaming micro-batch).
    """
    dim = spark.table(dim_table).filter(f"{surrogate_key} <> 0")
    versions = dim.select(F.col(business_key).alias("v_bk"), F.col(surrogate_key).alias("v_key"),
                          F.col("start_date").alias("v_start"), F.col("end_date").alias("v_end"))
    current = dim.filter("is_current").select(business_key, F.col(surrogate_key).alias("cur_key"))

    d = df.withColumn("_d", F.to_date(F.col(ts_col)))
    on = ((d[business_key] == versions["v_bk"])
          & (d["_d"] >= versions["v_start"])
          & (versions["v_end"].isNull() | (d["_d"] < versions["v_end"])))
    w = Window.partitionBy(row_key).orderBy(F.col("v_start").desc_nulls_last())
    picked = (d.join(versions, on, "left")
               .withColumn("_rn", F.row_number().over(w))
               .filter("_rn = 1")
               .drop("_rn", "v_bk", "v_start", "v_end", "_d"))
    return (picked.join(current, on=business_key, how="left")
                  .withColumn(surrogate_key, F.coalesce("v_key", "cur_key", F.lit(0).cast("long")))
                  .drop("v_key", "cur_key"))

def attach_user_key(df, ts_col, row_key, spark, db):
    return attach_scd2_key(df, ts_col, row_key, f"{db}.dim_users", "user_id", "user_key", spark)

def attach_product_key(df, ts_col, row_key, spark, db):
    return attach_scd2_key(df, ts_col, row_key, f"{db}.dim_products", "product_id", "product_key", spark)